# Calculating Power-law scaling in fluctuations of information

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [ ]:
CORPUS_NAME = ''

In [ ]:
# DATA_PATH = '/Volumes/Data-Repo/shapeoflang3/AF/obs'
DATA_PATH = '../../../updated-data/obs'
NULL_DATA_PATH = '../../../updated-data/null'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [ ]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

def lang_id(x):
    if 'xling-' in x:
        return x.split('-')[2]
    else:
        return 'eng'

def corpus_id(x):
    if 'xling-' in x:
        return x.split('-')[1]
    else:
        return x.split('-')[0]

In [ ]:
fs = [
    f for f in os.listdir(NULL_DATA_PATH)
    if (('xling-' in f) or ('callfriend' in f) or ('callhome' in f))
    and (not f.startswith('.'))
    and (f.endswith('.parquet'))
    and (CORPUS_NAME in f)
]
len(fs)

## Processing individual datasets

In [ ]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [ ]:
COEF_VAR = 'avar'

In [ ]:
time_col = 'mtau'
# time_col = 'n'

In [ ]:
df_scale, df_regression, k_docs, error_tracker = [], [], dict(), 0

for f in tqdm(fs):
    # print('===][===')
    # print(f.split('/')[-1])

    table = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f))

    df = table.read().to_pandas()
    df['self'] = df['x_speaker'] == df['y_speaker']
    df[COEF_VAR] = df[COEF_VAR] #/ df['n']

    # df['mtau'] = (df['min_y_turn_id'] + df['mtau']) - df['x_turn_id']
    # df[COEF_VAR] = np.sqrt(df[COEF_VAR])
    groups = df.groupby(by=['x_speaker', 'y_speaker'])
    # groups = df.groupby(by=['x_turn_id', 'y_speaker'])
    for labels, dfi in groups:
        self_other = dfi['self'].unique()[0]
        # dfi['min_y_turn_id'] = dfi.groupby('x_turn_id')['y_turn_id'].transform('min')

        av = dfi[COEF_VAR].values
        mtau = dfi[time_col].values
        sel = (av > 0) & (mtau > 1)

        try:
            if sel.sum() > 1:

                b, a = np.polyfit(
                    np.log(mtau[sel]),
                    np.log(av[sel]),
                    1
                )

                df_regression += [
                    {
                        'corpus': corpus_id(f),
                        'cid': f.replace('.parquet', ''),
                        'speaker': labels[0],
                        'self': self_other,
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

        except Exception:
            error_tracker += 1

df_regression = pd.DataFrame(df_regression)

In [ ]:
df_regression.isna().mean()

In [ ]:
np.isinf(df_regression['b']).mean()

In [ ]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [ ]:
df_regression.to_parquet(
    os.path.join(OUTPUTS_PATH,'null-all.parquet'),
    compression='snappy', engine='fastparquet'
)

## Corpus level analysis of results

In [ ]:
split_by = ['corpus', 'self']

In [ ]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [ ]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t', 'k']].values]

In [ ]:
df0.head(n=50)

In [ ]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'corpus-results.csv'),
    index=False, encoding='utf-8'
)

## Aggregate analysis

In [ ]:
split_by = ['self']

In [ ]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [ ]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t', 'k']].values]

In [ ]:
df0.head(n=20)

In [ ]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'null-results.csv'),
    index=False,
    encoding='utf-8'
)